# BRONZE → SILVER: Transformacion de Customers

Este notebook ejecuta el script de transformacion que:
1. Lee la capa Bronze (Parquet)
2. Aplica limpieza: trim de strings, deduplicacion, filtro de nulos, email a minusculas
3. Agrega columnas de metadatos
4. Guarda los datos limpios en la capa Silver (Parquet)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/agus/local/multihope


## 1. Verificar que la capa Bronze existe

In [2]:
BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'customers'

if not BRONZE_PATH.exists():
    raise FileNotFoundError(
        f'Bronze layer no encontrada en {BRONZE_PATH}.\n'
        'Ejecuta primero el notebook 01_raw_to_bronze_customers.ipynb'
    )
print(f'Bronze layer encontrada en: {BRONZE_PATH}')

Bronze layer encontrada en: /Users/agus/local/multihope/data/bronze/customers


## 2. Ejecutar el script BRONZE → SILVER

In [3]:
from src.bronze_to_silver.customers_transform import transform_customers

total_records = transform_customers()
print(f'\nTransformacion completada: {total_records} registros en Silver.')

26/04/07 09:02:36 WARN Utils: Your hostname, Agustins-MacBook-Pro-M4.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/04/07 09:02:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/agus/.ivy2/cache
The jars for the packages stored in: /Users/agus/.ivy2/jars
mysql#mysql-connector-java added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-caab5e28-2d14-4643-b6fb-ed764e010b58;1.0
	confs: [default]
	found mysql#mysql-connector-java;8.0.33 in central
	found com.mysql#mysql-connector-j;8.0.33 in central
	found com.google.protobuf#protobuf-java;3.21.9 in central
:: resolution report :: resolve 60ms :: artifacts dl 2ms
	:: modules in use:
	com.google.protobuf#protobuf-java;3.21.9 from central in [default]
	com.mysql#mysql-connector-j;8.0.33 from central in [default]
	mysql#mysql-connector-java;8.0.33 from central in [default]
	--------------------------------

:: loading settings :: url = jar:file:/Users/agus/local/multihope/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/04/07 09:02:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 09:02:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.



Transformacion completada: 10 registros en Silver.


## 3. Validar la capa Silver y revisar cuarentena

In [4]:
from src.utils.spark_session import create_spark_session

SILVER_PATH = PROJECT_ROOT / 'data' / 'silver' / 'customers'

spark = create_spark_session('notebook_silver_validation')
df_silver = spark.read.parquet(str(SILVER_PATH))

print('Schema Silver:')
df_silver.printSchema()

Schema Silver:
root
 |-- customer_id: integer (nullable = true)
 |-- identificacion: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telefono: string (nullable = true)
 |-- direccion: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- _loadtime: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_layer: string (nullable = true)



26/04/07 09:02:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
print(f'Total registros Silver: {df_silver.count()}')
df_silver.show(10, truncate=False)

Total registros Silver: 10
+-----------+--------------+-----------------------------+---------------------------+--------------+-----------------------------------------------------------------------------------------------+--------+-------------------+--------------------------+-------------+
|customer_id|identificacion|nombre                       |email                      |telefono      |direccion                                                                                      |estado  |_loadtime          |_ingested_at              |_source_layer|
+-----------+--------------+-----------------------------+---------------------------+--------------+-----------------------------------------------------------------------------------------------+--------+-------------------+--------------------------+-------------+
|2          |0700652068    |Dr. Estela Serrano           |ricardopantoja@gmail.com   |              |Boulevard Guatemala 748 521, San Salma los altos, ZAC 45032         

In [6]:
# Verificar que no hay nulos en customer_id (todas las filas deben haber pasado DQX)
from pyspark.sql import functions as F

null_count = df_silver.filter(F.col('customer_id').isNull()).count()
print(f'Registros con customer_id nulo: {null_count}')  # debe ser 0

# Revisar capa de cuarentena (filas que fallaron las reglas DQX)
QUARANTINE_PATH = PROJECT_ROOT / 'data' / 'quarantine' / 'customers'

if QUARANTINE_PATH.exists():
    df_quarantine = spark.read.parquet(str(QUARANTINE_PATH))
    print(f'\nRegistros en cuarentena: {df_quarantine.count()}')
    print('\nColumnas de audit DQX (_errors, _warnings):')
    df_quarantine.select('customer_id', 'nombre', 'email', '_errors', '_warnings').show(truncate=False)
else:
    print('\nNo se generó capa de cuarentena — todos los registros pasaron las validaciones DQX.')

Registros con customer_id nulo: 0

No se generó capa de cuarentena — todos los registros pasaron las validaciones DQX.


In [7]:
spark.stop()
print('SparkSession cerrada.')

SparkSession cerrada.
